# 2.3 **Train** and Compare Regularized Logistic Regression Models

## Model Cycle: The 5 Key Steps

### 1. Build the Model : Create the pipeline with regularization.  
### **2. Train the Model : Fit the model on the training data.**  
### 3. Generate Predictions : Use the trained model to make predictions.  
### 4. Evaluate the Model : Assess performance using evaluation metrics.  
### 5. Improve the Model : Tune hyperparameters for optimal performance.

## Introduction

In this notebook, we train the three regularized logistic regression models we built in the previous notebook and compare their behavior:

1. How do coefficients differ across regularization types?
2. Which features does L1 regularization select?
3. How do the models perform in cross-validation?

We also compare to the unregularized baseline from Course 2.

## 1. Load Dependencies and Data

In [1]:
# Data is in the project's data/ directory
# No Google Drive mount needed when running locally

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score

pd.options.display.max_columns = None

In [5]:
# Set up file paths
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
models_filepath = '/models/'

In [7]:
# Load training data
df_training = pd.read_csv(f'{project_path}{data_filepath}training.csv')

X_train = df_training
y_train = df_training['SEM_3_STATUS']

print(f"Training data: {X_train.shape[0]} samples")

Training data: 19844 samples


## 2. Load the Regularized Models

In [10]:
# Load regularized models from Course 3
l2_model = pickle.load(open(f'{project_path}{models_filepath}l2_ridge_logistic.pkl', 'rb'))
l1_model = pickle.load(open(f'{project_path}{models_filepath}l1_lasso_logistic.pkl', 'rb'))
elasticnet_model = pickle.load(open(f'{project_path}{models_filepath}elasticnet_logistic.pkl', 'rb'))

# Load baseline from Course 2 for comparison
baseline_model = pickle.load(open(f'{project_path}{models_filepath}baseline_logistic.pkl', 'rb'))

print("All models loaded successfully!")

All models loaded successfully!


In [11]:
# Create dictionary of all models for easy iteration
models = {
    'Baseline (No Penalty)': baseline_model,
    'L2 (Ridge)': l2_model,
    'L1 (Lasso)': l1_model,
    'ElasticNet': elasticnet_model
}

## 3. Train All Models

In [12]:
# Train all models
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print("Done!")

print("\nAll models trained successfully!")

Training Baseline (No Penalty)... Done!
Training L2 (Ridge)... Done!
Training L1 (Lasso)... Done!
Training ElasticNet... Done!

All models trained successfully!


## 4. Compare Coefficients

One of the key differences between regularization types is how they affect coefficient values. Let's examine this.

### 4.1 Coefficient Magnitudes

In [14]:
preprocessor = trained_models['Baseline (No Penalty)'].named_steps['preprocessor']
feature_names = preprocessor.get_feature_names_out()

# Clean up feature names for display
feature_names_clean = [name.split('__')[-1] for name in feature_names]
print(f"Number of features: {len(feature_names_clean)}")
print(f"\nFeatures: {feature_names_clean}")

Number of features: 16

Features: ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2', 'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2', 'GENDER_Male', 'RACE_ETHNICITY_Asian', 'RACE_ETHNICITY_Black or African American', 'RACE_ETHNICITY_Hispanic', 'RACE_ETHNICITY_Nonresident alien', 'RACE_ETHNICITY_Two or More Races', 'RACE_ETHNICITY_White', 'FIRST_GEN_STATUS_Continuing Generation', 'FIRST_GEN_STATUS_First Generation']


In [15]:
# Extract coefficients from each model
coef_data = []

for name, model in trained_models.items():
    classifier = model.named_steps['classifier']
    coefficients = classifier.coef_[0]

    for feat, coef in zip(feature_names_clean, coefficients):
        coef_data.append({
            'Model': name,
            'Feature': feat,
            'Coefficient': coef
        })

coef_df = pd.DataFrame(coef_data)
coef_df.head(10)

,Model,Feature,Coefficient
0,Baseline (No Penalty),HS_GPA,0.236373
1,Baseline (No Penalty),GPA_1,-0.974851
2,Baseline (No Penalty),GPA_2,0.572466
3,Baseline (No Penalty),DFW_RATE_1,2.884175
4,Baseline (No Penalty),DFW_RATE_2,1.813520
5,Baseline (No Penalty),UNITS_ATTEMPTED_1,-0.125248
6,Baseline (No Penalty),UNITS_ATTEMPTED_2,-0.152224
7,Baseline (No Penalty),GENDER_Male,0.030340
8,Baseline (No Penalty),RACE_ETHNICITY_Asian,-0.640901
9,Baseline (No Penalty),RACE_ETHNICITY_Black or African American,-0.576379


In [16]:
# Create coefficient comparison visualization
fig = px.bar(
    coef_df,
    x='Feature',
    y='Coefficient',
    color='Model',
    barmode='group',
    title='Coefficient Comparison Across Regularization Types',
    height=500
)

fig.update_xaxes(tickangle=45)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02))
fig.show()

In [17]:
# Compare coefficient magnitudes (L2 norm per model)
coef_norms = {}

for name, model in trained_models.items():
    classifier = model.named_steps['classifier']
    coefficients = classifier.coef_[0]
    l2_norm = np.sqrt(np.sum(coefficients**2))
    l1_norm = np.sum(np.abs(coefficients))
    coef_norms[name] = {'L2 Norm': l2_norm, 'L1 Norm': l1_norm}

norms_df = pd.DataFrame(coef_norms).T
print("Coefficient Norms (measure of model complexity):")
display(norms_df)

Coefficient Norms (measure of model complexity):


,L2 Norm,L1 Norm
Baseline (No Penalty),3.784026,9.617578
L2 (Ridge),3.475141,7.963048
L1 (Lasso),3.480915,7.861598
ElasticNet,3.478591,7.903257


**Interpretation:**
- The **baseline model** (no regularization) typically has the largest coefficient norms
- **L2 regularization** shrinks coefficients proportionally
- **L1 regularization** shrinks some coefficients to exactly zero, reducing overall norm
- **ElasticNet** combines both behaviors

### 4.2 Feature Selection with L1

In [18]:
# Examine which features L1 keeps vs zeros out
l1_classifier = trained_models['L1 (Lasso)'].named_steps['classifier']
l1_coefficients = l1_classifier.coef_[0]

l1_selection = pd.DataFrame({
    'Feature': feature_names_clean,
    'Coefficient': l1_coefficients,
    'Selected': l1_coefficients != 0
}).sort_values('Coefficient', key=abs, ascending=False)

print("L1 (Lasso) Feature Selection:")
print(f"Total features: {len(l1_selection)}")
print(f"Selected (non-zero): {l1_selection['Selected'].sum()}")
print(f"Eliminated (zero): {(~l1_selection['Selected']).sum()}")
print("\nFeatures by importance:")
display(l1_selection)

L1 (Lasso) Feature Selection:
Total features: 16
Selected (non-zero): 16
Eliminated (zero): 0

Features by importance:


,Feature,Coefficient,Selected
3,DFW_RATE_1,2.798056,True
4,DFW_RATE_2,1.670257,True
1,GPA_1,-0.857020,True
8,RACE_ETHNICITY_Asian,-0.477608,True
9,RACE_ETHNICITY_Black or African American,-0.412407,True
10,RACE_ETHNICITY_Hispanic,-0.309674,True
2,GPA_2,0.287853,True
0,HS_GPA,0.233970,True
6,UNITS_ATTEMPTED_2,-0.223919,True
11,RACE_ETHNICITY_Nonresident alien,-0.176005,True


In [19]:
# Visualize L1 feature selection
fig = go.Figure()

colors = ['green' if s else 'lightgray' for s in l1_selection['Selected']]

fig.add_trace(go.Bar(
    x=l1_selection['Feature'],
    y=l1_selection['Coefficient'],
    marker_color=colors
))

fig.update_layout(
    title='L1 (Lasso) Coefficients: Green = Selected, Gray = Eliminated',
    xaxis_tickangle=45,
    height=400
)

fig.show()

## 5. Cross-Validation Comparison

Let's compare model performance using cross-validation, similar to what we did in Course 2.

In [20]:
# Define cross-validation scorer for minority class
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scorer = make_scorer(f1_score, pos_label='N')
precision_scorer = make_scorer(precision_score, pos_label='N')
recall_scorer = make_scorer(recall_score, pos_label='N')

In [21]:
# Run cross-validation for all models
cv_results = []

for name, model in models.items():  # Use original (untrained) models
    print(f"Cross-validating {name}...", end=" ")

    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=f1_scorer)
    precision_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=precision_scorer)
    recall_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=recall_scorer)

    cv_results.append({
        'Model': name,
        'F1 Mean': f1_scores.mean(),
        'F1 Std': f1_scores.std(),
        'Precision Mean': precision_scores.mean(),
        'Precision Std': precision_scores.std(),
        'Recall Mean': recall_scores.mean(),
        'Recall Std': recall_scores.std()
    })
    print("Done!")

cv_df = pd.DataFrame(cv_results)
print("\nCross-Validation Results (Positive class: 'N' - Students who leave):")
display(cv_df)

Cross-validating Baseline (No Penalty)... Done!
Cross-validating L2 (Ridge)... Done!
Cross-validating L1 (Lasso)... Done!
Cross-validating ElasticNet... Done!

Cross-Validation Results (Positive class: 'N' - Students who leave):


,Model,F1 Mean,F1 Std,Precision Mean,Precision Std,Recall Mean,Recall Std
0,Baseline (No Penalty),0.494563,0.020191,0.724738,0.026478,0.376555,0.026692
1,L2 (Ridge),0.493373,0.009295,0.391301,0.007388,0.667794,0.018924
2,L1 (Lasso),0.494215,0.009148,0.392244,0.007589,0.668174,0.018866
3,ElasticNet,0.493798,0.009045,0.391719,0.007457,0.668174,0.018866


In [22]:
# Visualize CV results
fig = make_subplots(rows=1, cols=3, subplot_titles=('F1 Score', 'Precision', 'Recall'))

for i, metric in enumerate(['F1', 'Precision', 'Recall'], 1):
    fig.add_trace(
        go.Bar(
            x=cv_df['Model'],
            y=cv_df[f'{metric} Mean'],
            error_y=dict(type='data', array=cv_df[f'{metric} Std']),
            name=metric,
            showlegend=False
        ),
        row=1, col=i
    )

fig.update_layout(
    height=400,
    title_text='Cross-Validation Results by Regularization Type'
)
fig.update_xaxes(tickangle=45)
fig.show()

## 6. Save Trained Models

In [ ]:
# Save trained models
for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    filepath = f'{course3_models}{filename}_trained.pkl'
    pickle.dump(model, open(filepath, 'wb'))
    print(f"Saved: {filepath}")

## 7. Summary

In this notebook, we trained and compared regularized logistic regression models:

### Key Findings

1. **Coefficient Shrinkage**: Regularization reduces coefficient magnitudes compared to the baseline

2. **Feature Selection**: L1 (Lasso) naturally performs feature selection by zeroing out less important features

3. **Cross-Validation Performance**: Regularized models often show similar or improved performance with better generalization

### Comparison Summary

In [23]:
# Final comparison table
summary = cv_df[['Model', 'F1 Mean', 'Precision Mean', 'Recall Mean']].copy()
summary = summary.round(3)
print("Cross-Validation Summary:")
display(summary)

Cross-Validation Summary:


,Model,F1 Mean,Precision Mean,Recall Mean
0,Baseline (No Penalty),0.495,0.725,0.377
1,L2 (Ridge),0.493,0.391,0.668
2,L1 (Lasso),0.494,0.392,0.668
3,ElasticNet,0.494,0.392,0.668


### Next Steps

In the next notebook, we will:
1. Tune the regularization strength (`C` parameter) using GridSearch
2. Find optimal hyperparameters for each regularization type
3. Evaluate final models on the test set

**Proceed to:** `2.4 Tune Regularization Hyperparameters`